In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from box_embeddings.parameterizations import BoxTensor, MinDeltaBoxTensor
from box_embeddings.modules.intersection import GumbelIntersection
from box_embeddings.modules.volume import SoftVolume

# --- 1. SETUP DELL'INTERA ARCHITETTURA ---
k = 5
num_dims = 2
feature_dim = 64
batch_size = 4

torch.manual_seed(42)
features = torch.randn(batch_size, feature_dim)

# I nostri estrattori
projectors = nn.ModuleList([nn.Linear(feature_dim, 2 * num_dims) for _ in range(k)])
prob_predictors = nn.ModuleList([nn.Linear(2 * num_dims, 1) for _ in range(k)])

# --- I DUE NUOVI CLASSIFICATORI PER IL TASK FINALE ---
# 1. Prende i box scalati appiattiti: k concetti * 2 coordinate (z,Z) * num_dims
clf_boxes = nn.Linear(in_features=k * 2 * num_dims, out_features=1)

# 2. Prende la matrice k x k appiattita
clf_relations = nn.Linear(in_features=k * k, out_features=1)

# Passiamo tutti i parametri all'ottimizzatore
optimizer = torch.optim.Adam(
    list(projectors.parameters()) + list(prob_predictors.parameters()) +
    list(clf_boxes.parameters()) + list(clf_relations.parameters()), 
    lr=0.01
)

gumbel_intersection = GumbelIntersection(intersection_temperature=0.1)
soft_volume = SoftVolume(volume_temperature=0.5)

# --- 2. GROUND TRUTH (Tre livelli di supervisione!) ---
# A. Gerarchia
supervisions = [
    (0, 1, 1.0), 
    (0, 2, 1.0), 
    (1, 2, 0.0), 
    (2, 1, 0.0),
    (1, 3, 1.0), 
    (1, 4, 1.0), 
    (3, 4, 0.0), 
    (4, 3, 0.0),
]

# B. Attivazione dei concetti
concept_labels = torch.tensor([
    [1.0, 1.0, 0.0, 1.0, 0.0],
    [1.0, 0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0, 1.0],
], dtype=torch.float32)

# C. Etichette del TASK FINALE (Nuovo!) - Es: Classificazione binaria dell'immagine
task_labels = torch.tensor([[1.0], [0.0], [0.0], [1.0]], dtype=torch.float32)

# --- 3. LOOP DI ADDESTRAMENTO END-TO-END ---
epochs = 350 

for epoch in range(epochs):
    optimizer.zero_grad()
    
    boxes = []
    logits = []
    
    # 1. Creazione Box e Probabilità
    for i in range(k):
        theta_i = projectors[i](features)
        box_i = MinDeltaBoxTensor(theta_i.view(batch_size, 2, num_dims))
        boxes.append(box_i)
        
        coords = torch.cat([box_i.z, box_i.Z], dim=-1)
        logit_i = prob_predictors[i](coords).squeeze(-1)
        logits.append(logit_i)
        
    logits_tensor = torch.stack(logits, dim=1)
    concept_probs = torch.sigmoid(logits_tensor)
    
    # 2. COSTRUZIONE MATRICE KxK DELLE PROBABILITÀ CONDIZIONATE
    # Creiamo la matrice in modo differenziabile usando liste e torch.stack
    matrix_rows = []
    for i in range(k): # Target (Contenitore)
        row = []
        for j in range(k): # Source (Da contenere)
            int_box = gumbel_intersection(boxes[i], boxes[j])
            prob = torch.exp(soft_volume(int_box) - soft_volume(boxes[j]))
            row.append(torch.clamp(prob, 1e-6, 1.0 - 1e-6))
        matrix_rows.append(torch.stack(row, dim=1))
        
    # Shape: (batch_size, k_target, k_source)
    cond_prob_matrix = torch.stack(matrix_rows, dim=1) 
    
    # 3. CREAZIONE BOX SCALATI (GATING)
    scaled_coords_list = []
    for i in range(k):
        p = concept_probs[:, i].unsqueeze(-1) # [batch, 1]
        z_scaled = boxes[i].z * p
        Z_scaled = boxes[i].Z * p
        # Appiattiamo le coordinate del box scalato per il classificatore
        scaled_coords_list.append(torch.cat([z_scaled, Z_scaled], dim=-1))
        
    # --- 4. CLASSIFICAZIONE DEL TASK FINALE ---
    # Uniamo tutti i box scalati: shape (batch_size, k * 4)
    flat_scaled_boxes = torch.cat(scaled_coords_list, dim=-1)
    
    # Appiattiamo la matrice KxK: shape (batch_size, k * k)
    flat_relation_matrix = cond_prob_matrix.view(batch_size, k * k)
    
    # Passiamo gli input ai due classificatori
    task_logit_from_boxes = clf_boxes(flat_scaled_boxes)
    task_logit_from_relations = clf_relations(flat_relation_matrix)
    
    # Uniamo i due segnali (sommandoli) per ottenere la predizione finale
    final_task_logit = task_logit_from_boxes + task_logit_from_relations
    final_task_prob = torch.sigmoid(final_task_logit)
    
    # --- 5. CALCOLO DELLE LOSS ---
    # A. Loss Attivazione
    activation_loss = F.binary_cross_entropy(concept_probs, concept_labels)
    
    # B. Loss Gerarchica (Ora molto più elegante estraendo dalla matrice!)
    hierarchy_loss = 0.0
    for target_id, source_id, target_prob in supervisions:
        pred_prob = cond_prob_matrix[:, target_id, source_id]
        target_tensor = torch.full((batch_size,), target_prob, dtype=torch.float32)
        hierarchy_loss += F.binary_cross_entropy(pred_prob, target_tensor)
        
    # C. Loss Anti-Collasso Volumi
    vol_loss = 0.0
    for i in range(1, k):
        vol_loss -= 0.1 * soft_volume(boxes[i]).mean()
        
    # D. LOSS DEL TASK FINALE (Nuova!)
    task_loss = F.binary_cross_entropy(final_task_prob, task_labels)

    # LOSS TOTALE
    final_loss = activation_loss + hierarchy_loss + vol_loss + (2.0 * task_loss) # Diamo più peso al task finale
    
    final_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Ep {epoch+1} | TaskL: {task_loss.item():.3f} | ActL: {activation_loss.item():.3f} | HierL: {hierarchy_loss.item():.3f}")

# Valutazione
with torch.no_grad():
    print("\n--- Previsioni Task Finale ---")
    for b in range(batch_size):
        print(f"Immagine {b}: Prevista={final_task_prob[b].item():.3f} | Reale={task_labels[b].item()}")

AllenNLP not available. Registrable won't work.


Ep 50 | TaskL: 0.018 | ActL: 0.347 | HierL: 0.083
Ep 100 | TaskL: 0.004 | ActL: 0.082 | HierL: 0.094
Ep 150 | TaskL: 0.002 | ActL: 0.021 | HierL: 0.060
Ep 200 | TaskL: 0.001 | ActL: 0.011 | HierL: 0.047
Ep 250 | TaskL: 0.001 | ActL: 0.007 | HierL: 0.039
Ep 300 | TaskL: 0.000 | ActL: 0.005 | HierL: 0.033
Ep 350 | TaskL: 0.000 | ActL: 0.004 | HierL: 0.030

--- Previsioni Task Finale ---
Immagine 0: Prevista=1.000 | Reale=1.0
Immagine 1: Prevista=0.000 | Reale=0.0
Immagine 2: Prevista=0.001 | Reale=0.0
Immagine 3: Prevista=1.000 | Reale=1.0


## Interpretation

In [8]:
def explain_prediction(image_index, features, concept_names=None):
    """
    Spiega la decisione del modello per una singola immagine del batch.
    """
    if concept_names is None:
        concept_names = [f"C{i}" for i in range(k)]
        
    print(f"\n{'='*50}")
    print(f" SPIEGAZIONE DECISIONE PER L'IMMAGINE {image_index}")
    print(f"{'='*50}")
    
    with torch.no_grad():
        # 1. Ricalcoliamo il forward pass solo per questa immagine
        feat = features[image_index].unsqueeze(0) # Shape: (1, feature_dim)
        
        boxes = []
        logits = []
        for i in range(k):
            theta_i = projectors[i](feat)
            box_i = MinDeltaBoxTensor(theta_i.view(1, 2, num_dims))
            boxes.append(box_i)
            
            coords = torch.cat([box_i.z, box_i.Z], dim=-1)
            logits.append(prob_predictors[i](coords).squeeze(-1))
            
        logits_tensor = torch.stack(logits, dim=1)
        concept_probs = torch.sigmoid(logits_tensor)[0] # Probabilità per l'immagine
        
        # 2. Ricalcoliamo la Matrice delle Relazioni (P(Ci | Cj))
        cond_prob_matrix = torch.zeros((k, k))
        for i in range(k):
            for j in range(k):
                int_box = gumbel_intersection(boxes[i], boxes[j])
                prob = torch.exp(soft_volume(int_box) - soft_volume(boxes[j]))
                cond_prob_matrix[i, j] = torch.clamp(prob, 1e-6, 1.0 - 1e-6)[0]
                
        # 3. Ricalcoliamo i Box Scalati
        scaled_coords_list = []
        for i in range(k):
            p = concept_probs[i]
            z_scaled = boxes[i].z[0] * p
            Z_scaled = boxes[i].Z[0] * p
            scaled_coords_list.append(torch.cat([z_scaled, Z_scaled], dim=-1))
            
        flat_scaled_boxes = torch.cat(scaled_coords_list, dim=-1).unsqueeze(0)
        flat_relation_matrix = cond_prob_matrix.view(1, k * k)
        
        # 4. Predizione Finale
        logit_boxes = clf_boxes(flat_scaled_boxes)
        logit_rels = clf_relations(flat_relation_matrix)
        final_prob = torch.sigmoid(logit_boxes + logit_rels).item()
        
        print(f"PREDIZIONE TASK FINALE: {final_prob:.4f} (1.0 = Positivo, 0.0 = Negativo)\n")
        
        # --- ESTRAZIONE DEI CONTRIBUTI ---
        
        print("--- 1. CONTRIBUTI DEI SINGOLI CONCETTI (Presenza & Geometria) ---")
        box_weights = clf_boxes.weight[0] # Shape: (k * 4)
        concept_contributions = []
        
        for i in range(k):
            # Estraiamo le 4 coordinate scalate del concetto e i 4 pesi associati
            coords_i = scaled_coords_list[i]
            weights_i = box_weights[i*4 : (i+1)*4]
            
            # Il contributo è il prodotto scalare: sum(coordinata * peso)
            contrib = torch.dot(coords_i, weights_i).item()
            concept_contributions.append((concept_names[i], contrib, concept_probs[i].item()))
            
        # Ordiniamo dal contributo più positivo a quello più negativo
        concept_contributions.sort(key=lambda x: x[1], reverse=True)
        for name, contrib, prob in concept_contributions:
            segno = "+" if contrib > 0 else ""
            print(f"{name:5} | Attivazione: {prob:.2f} | Contributo al Task: {segno}{contrib:.4f}")
            
            
        print("\n--- 2. CONTRIBUTI DELLE RELAZIONI LOGICHE (P(Target | Source)) ---")
        rel_weights = clf_relations.weight[0] # Shape: (k * k)
        relation_contributions = []
        
        for i in range(k):
            for j in range(k):
                if i == j: continue # Ignoriamo la relazione di un concetto con se stesso (sempre 1.0)
                
                idx = i * k + j
                weight = rel_weights[idx].item()
                prob = cond_prob_matrix[i, j].item()
                
                contrib = prob * weight
                # Filtriamo solo le relazioni che hanno una probabilità e un peso rilevanti
                if abs(contrib) > 0.01: 
                    relation_contributions.append((concept_names[i], concept_names[j], contrib, prob))
                    
        # Ordiniamo per contributo
        relation_contributions.sort(key=lambda x: x[2], reverse=True)
        for target, source, contrib, prob in relation_contributions:
            segno = "+" if contrib > 0 else ""
            print(f"P({target} | {source}) = {prob:.2f} | Contributo al Task: {segno}{contrib:.4f}")

# Esempio di utilizzo:
# Spieghiamo la decisione per la prima immagine del batch
nomi_concetti = ["C1", "C2", "C3", "C4", "C5"] # Sostituisci con i tuoi
explain_prediction(image_index=0, features=features, concept_names=nomi_concetti)


 SPIEGAZIONE DECISIONE PER L'IMMAGINE 0
PREDIZIONE TASK FINALE: 1.0000 (1.0 = Positivo, 0.0 = Negativo)

--- 1. CONTRIBUTI DEI SINGOLI CONCETTI (Presenza & Geometria) ---
C2    | Attivazione: 0.99 | Contributo al Task: +11.8791
C1    | Attivazione: 1.00 | Contributo al Task: +5.2930
C4    | Attivazione: 0.99 | Contributo al Task: +0.3063
C5    | Attivazione: 0.00 | Contributo al Task: +0.0001
C3    | Attivazione: 0.00 | Contributo al Task: +0.0000

--- 2. CONTRIBUTI DELLE RELAZIONI LOGICHE (P(Target | Source)) ---
P(C4 | C2) = 0.37 | Contributo al Task: -0.0501
P(C5 | C2) = 0.48 | Contributo al Task: -0.0535
P(C2 | C1) = 0.54 | Contributo al Task: -0.0790
P(C1 | C4) = 1.00 | Contributo al Task: -0.0971
P(C3 | C1) = 0.35 | Contributo al Task: -0.1678
P(C1 | C5) = 1.00 | Contributo al Task: -0.1996
P(C1 | C2) = 1.00 | Contributo al Task: -0.3903
P(C2 | C5) = 1.00 | Contributo al Task: -0.4708
P(C2 | C4) = 1.00 | Contributo al Task: -0.4780
P(C1 | C3) = 1.00 | Contributo al Task: -0.5363